In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load the text data
print("Loading dataset...")
df = pd.read_csv("../data/cleaned_arxiv_papers.csv") # Adjust path if needed based on your folder structure

# 2. Load the mathematical brain (Takes 1 second instead of 26 minutes!)
print("Loading embeddings...")
embeddings = np.load("../data/arxiv_embeddings.npy")

# 3. Load the model (We need this to convert the user's search query into a vector)
print("Loading model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("System Ready! Embeddings shape:", embeddings.shape)

Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: '../data/cleaned_arxiv_papers.csv'

In [ ]:
print(embeddings.shape)
print(embeddings.dtype)
embeddings



### **1. Why is the dtype `float32`? (The Decimal Point)**

When you look at `embedding.dtype`, it returns `float32`. This means every single number inside your massive array is a 32-bit floating-point number (a decimal number like `0.04561` or `-0.89213`).

* **Why decimals and not whole numbers?** Neural networks capture *subtle* semantic meanings. If they used integers (1, 2, 3), they couldn't capture fine details. A paper might be `0.78` similar to "AI" and `0.21` similar to "Biology". You need decimals to plot these exact coordinates in space.
* **Why `32-bit`?** In Deep Learning, `float32` is the golden industry standard. It gives the model plenty of mathematical precision to represent complex thoughts, but it takes up exactly half the computer memory as a `float64`. This is what allows your computer to calculate the Cosine Similarity so incredibly fast.

### **2. Why are there exactly 384 columns?**

Look at the shape: `(15000, 384)`.

* **15,000** is your number of rows (you successfully encoded 15,000 research papers).
* **384** is your number of columns. As we discussed in **Section 3 of your `dl_nlp_basics.md` notes**, this is the **dimensionality** of your embedding.
* **The "Why":** The specific neural network you downloaded (`all-MiniLM-L6-v2`) was hard-coded by its creators to output exactly 384 numbers. To the AI, every single one of those 384 columns represents a highly abstract "feature" of the text.
* Column 1 might measure how "academic" the text is.
* Column 2 might measure how strongly it relates to "algorithms".
* Column 3 might track "medical context".



*(Note: We can't actually read the columns like that as humans, but mathematically, that is how the neural network organizes the meaning).*

### **Interview Context:**

If an interviewer asks: *"Explain the output shape and type of your embedding matrix."*

**Your Answer:** *"My embedding matrix has a shape of 15,000 by 384, meaning I processed 15,000 documents, and the `MiniLM` model compressed the semantic meaning of each document into a 384-dimensional dense vector. The data type is `float32`, which is standard for neural network weights and embeddings because it perfectly balances computational speed, memory efficiency, and mathematical precision for downstream tasks like Cosine Similarity."*

In [ ]:
import faiss

# 1. We must make a copy of the embeddings first because FAISS normalizes "in-place"
# (Meaning it permanently alters the original variable)
faiss_embeddings = embeddings.copy()

# 2. Apply L2 Normalization
faiss.normalize_L2(faiss_embeddings)

print("Embeddings normalized! Ready for the FAISS Index.")


### **FAISS L2 Normalization & Cosine Similarity**

**The Objective**
To normalize your 384-dimensional document embeddings (forcing their vector lengths to exactly 1) so that FAISS can use its lightning-fast Inner Product engine to calculate mathematically perfect Cosine Similarity scores.

**The Explanation**

* **Why are we normalizing?** In semantic search, the "meaning" of a sentence is stored in the *direction* the vector points, not how long the vector is. By scaling every single vector down to an exact length of 1, we remove any bias caused by vector size and force the math to look *only* at the angle.
* **Why specifically "L2"? (The Math Hack):** FAISS is incredibly fast but **does not** have a built-in Cosine Similarity function. However, it does have a blazing-fast Inner Product (Dot Product) function. The mathematical rule is: `L2 Normalization + Inner Product = Cosine Similarity`. By running `faiss.normalize_L2`, you are hacking FAISS to output the exact Cosine Similarity scores you want while keeping its extreme C++ speed.
* **The Output Range:** Because of this math hack, the scores FAISS will eventually return to you will fall into the standard Cosine Similarity range of **-1.0 to 1.0**:
* **1.0 (Maximum Score):** The vectors point in the exact same direction (identical semantic meaning).
* **0.0 (Neutral Score):** The vectors are perpendicular (unrelated meaning).
* **-1.0 (Minimum Score):** The vectors point in exact opposite directions.



*(Note: In real-world text data, you will almost never see negative scores. Most of your search results will score between `0.0` and `1.0`)*



In [ ]:
# index = faiss.IndexFlatIP(384)
# index.add(faiss_embeddings)
# index

In [ ]:
import os

# Define the exact path to your data folder
index_path = "../data/paper_faiss.index"

if os.path.exists(index_path):
    print("Loading existing FAISS index")
    index = faiss.read_index(index_path)
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(384)
    index.add(embeddings)

    # Save it using the new path variable
    faiss.write_index(index, index_path)
    print("FAISS index saved successfully to the data folder!")

### **FAISS Index Initialization**

**The Objective**
To create the empty structural container (the database "Index") in your computer's RAM, define the mathematical rules it will use to compare vectors, and tell it exactly what size those vectors will be.


**The Explanation**

* **What is `Index`?** In FAISS, an "Index" is essentially the database object. It is the highly optimized C++ data structure that will hold all your research paper embeddings in memory so they can be searched instantly.
* **What is `Flat`?** This tells FAISS how to search. "Flat" means it stores the vectors in a flat, uncompressed array and performs an **Exact Search** (brute-force). It will not skip any vectors; it will compare your search query to every single paper. *(Since you are only using a subset of about 50,000 papers, a Flat index is still incredibly fast. If you had 15 million papers, we would change this to `IVFFlat` to cluster the data).*
* **What is `IP`?** This stands for **Inner Product**. Because you just ran the L2 Normalization step in the previous cell, setting the index to `IP` is the final step of our "Math Hack." It tells the index to use the lightning-fast dot product multiplier, which will perfectly output Cosine Similarity scores.
* **What is `384`?** This is the dimensionality `(d)` of your vectors. FAISS needs to allocate the exact amount of memory required before it accepts any data. You are telling it: *"Get ready, I am about to hand you arrays that have exactly 384 floating-point numbers in them."*
* **Why is the index currently empty?** Right now, you just bought the bookshelf and assembled it, but you haven't put any books on it yet. You have defined the *rules* and the *size* of the database, but it currently holds `0` vectors.

The very next step is to actually take your `faiss_embeddings` matrix and "add" it to this empty index! Let me know when you are ready for that code.

### **Encoding the User Query**

**The Objective**
To take the raw text that a user types into the search bar and convert it into the exact same 384-dimensional mathematical language that our database of research papers uses.


In [ ]:
query = "deep learning for medical image analysis"
query_embedding = model.encode(query)
print(query_embedding.shape)


**The Explanation**

* **`query = ...`**: This is simply simulating a user typing a search term into your final application.
* **`model.encode(query)`**: You are passing that raw text through the exact same `SentenceTransformer` (`all-MiniLM-L6-v2`) that you used to encode your 15,000 research papers. It is crucial to use the exact same model so that the query and the papers exist in the exact same mathematical space.
* **The Output `(384,)**`: This confirms the model successfully compressed your search phrase into a single, flat 1D array containing 384 floating-point numbers.

**Important Note for the Next Step:**
Remember the `ValueError` we hit way back when we first used `cosine_similarity`? Because FAISS (just like `sklearn`) expects to compare *matrices* (2D arrays), passing a flat `(384,)` array will cause an error. Before we search the FAISS index, we will need to quickly reshape this query vector into a 2D matrix of shape `(1, 384)`!

In [ ]:
query_embedding = model.encode([query])
# query_embedding = model.encode(query).reshape(1,-1)
print(query_embedding.shape)
print(query_embedding)

In [ ]:
faiss.normalize_L2(query_embedding)
query_embedding

In [ ]:
D, I = index.search(query_embedding, 5)

print("Scores (D):", D)
print("Indices (I):", I)

In [ ]:
print(df.iloc[26063]['title'])

This is the grand finale of your search engine backend, bro! You just successfully queried your FAISS database.

Here is the exact breakdown for your notes, including exactly what that `5` means!

### **Executing the FAISS Search**

**The Objective**
To search the FAISS index using the normalized user query, instantly finding the mathematical coordinates of the most relevant research papers and returning their similarity scores and row numbers.

**The Code**

```python
# Search the index!
D, I = index.search(query_embedding, 5)

print("Scores (D):", D)
print("Indices (I):", I)

```

**The Explanation**

* **What is the `5`? (Top-K):** In Machine Learning, this is known as `k`, which stands for how many results you want to retrieve. By passing `5`, you are telling FAISS: *"Find the top 5 closest vectors to my query."* If you wanted to show 10 results on a webpage, you would change this to `10`.
* **What is `D`? (Distances / Scores):** Look at your output: `[[0.72542226 0.7170659  0.71233034 0.6807244  0.6798818 ]]`. Because we used our L2 Normalization + Inner Product hack, these are your **Cosine Similarity Scores**!
* The system found a paper with a `~0.72` match score.
* Notice how they are automatically sorted from highest to lowest. FAISS did all the heavy sorting logic for you!


* **What is `I`? (Indices / Row Numbers):** Look at your output: `[[26063 37161 18030 10466 29766]]`. This is the most important part! FAISS is saying:
* *"The absolute best match is the paper sitting at **Row 37161** in your dataset."*
* *"The second best match is at **Row 37161**."*



### **Next Steps (The Final Piece of the Puzzle)**

Right now, you just have a row number . A user doesn't care about row numbers; they want to read the actual title of the research paper!

Your very last step to complete this project is to take that `I` array, go back to your pandas DataFrame (`df`), and print out the title and abstract for row

now lets make a finction to just returen top 5 papers and we dont neet to write print [df.iloca] everutime

In [ ]:
def search_paper(query,k=5):
    query_embedding=model.encode([query])
    faiss.normalize_L2(query_embedding)
    D,I=index.search(query_embedding,k)
    for score,idx in zip(D[0],I[0]):
        print("Similarity Score: ",score)
        print("Title: ",df.iloc[idx]['title'])
        print("Abstract:" ,df.iloc[idx]['abstract'][:500])
        print() # for space between two papers
    return D,I

Because you only passed 1 query, D and I variables are technically 2D arrays that look like this: [[0.68, 0.67...]]. Adding [0] extracts just that inner list of 5 numbers. The zip() function is a brilliant Python tool that lets us loop through the scores list and the indices list at the exact same time.

In [ ]:
D,I=search_paper(query)
# print(D)
# print(I)

In [ ]:
import transformers
from transformers import pipeline
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",  # 300MB instead of 1.6GB
    device=0
)

In [ ]:
type(summarizer)

In [ ]:
summary=summarizer(df.iloc[10466]['abstract'],max_length=120,min_length=40)
print(summary)
# print(summary[0].dtype)
print(type(summary[0]))
print(summary[0]["summary_text"])

`.dtype` only works on NumPy arrays and Pandas Series/DataFrames. It has no meaning on a dictionary hence `AttributeError: 'dict' object has no attribute 'dtype'`. You were probably testing what type the output was, but `type(summary[0])` is the right way to check that, not `.dtype`.

### **Generating AI Summaries from Search Results**

**The Objective**
To loop through the top FAISS search results, fetch the original research paper details from your pandas DataFrame, and use the BART AI model to dynamically read and summarize the massive academic abstracts into bite-sized, readable paragraphs.



In [ ]:
for score, idx in zip(D[0], I[0]):
    print("Similarity score:", score)
    print("Title:", df.iloc[idx]["title"])
    print("Abstract (Preview):", df.iloc[idx]["abstract"][:500] + "...\n")

    # Generate the summary using the AI pipeline
    summary = summarizer(df.iloc[idx]["abstract"], max_length=120, min_length=40)

    # print(summary) # We can comment this out later, it shows the raw dictionary
    print("AI SUMMARY:")
    print(summary[0]["summary_text"])
    print("-" * 80) # Adds a nice dividing line between papers

### **Systematic Code Breakdown**

* **`for score, idx in zip(D[0], I[0]):`**
* `D[0]` holds your top 5 Cosine Similarity scores. `I[0]` holds the top 5 pandas row numbers (indices).
* The `zip()` function acts like a zipper on a jacket—it pairs them up so you can loop through the score and the row number at the exact same time.


* **`df.iloc[idx]["title"]`**
* You are using pandas "integer location" (`iloc`) to jump directly to the winning row number in your dataset, and extracting the text from the `title` column.


* **`df.iloc[idx]["abstract"][:500]`**
* This grabs the abstract text, but the `[:500]` is a Python string slice. It tells Python to only print the first 500 characters. Academic abstracts are massive; if you print the whole thing, it will flood your VS Code terminal!


* **`summary = summarizer(..., max_length=120, min_length=40)`**
* This is where the magic happens. You pass the *entire* abstract into your Hugging Face `summarizer` pipeline.
* **The Guardrails:** By setting `max_length=120` and `min_length=40`, you are forcing the AI to give you a "Goldilocks" summary—not too long (preventing rambling) and not too short (preventing a useless 3-word answer).


* **`print(summary)` vs `print(summary[0]["summary_text"])**`
* If you just print the raw `summary` variable, Hugging Face returns an ugly list containing a dictionary that looks like this: `[{'summary_text': 'This paper explores deep learning...'}]`.
* To get just the clean, human-readable English, you have to drill into it: `[0]` gets inside the list, and `["summary_text"]` gets the value out of the dictionary.

In [ ]:
def search_and_summarize(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):
        print("Similarity score:", score)
        print("Title:", df.iloc[idx]["title"])
        print("Abstract:", df.iloc[idx]["abstract"][:500])
        print()

        # This MUST be indented inside the loop to summarize every paper!
        summary = summarizer(df.iloc[idx]["abstract"], max_length=120, min_length=40, do_sample=False)
        print(summary)
        print("AI SUMMARY:", summary[0]["summary_text"])
        print("-" * 80) # Added a divider line to keep the output readable

In [ ]:
search_and_summarize("Deep learning in medical science",k=5)

In [ ]:
from keybert import KeyBERT

# kw_model=keybert()
kw_model=KeyBERT(model=model) # as by defualt it may take some another model, so we want the same model to create embeeding the earlier one we used in sentence transofmrer we will use same model
type(kw_model)

The KeyBERT Error (TypeError: 'module' object is not callable)
The Reason:
In Python, capitalization matters immensely! When you typed import keybert, you imported the entire library (the module). But to actually create the model, you need to call the specific Class inside that library, which is spelled with capital letters (KeyBERT). You tried to "call" the whole folder instead of the specific tool inside of it.

The Fix:
You just need to import the class specifically. Change your cell to this:

In [ ]:
text=df.iloc[26063]['abstract']
keywords=kw_model.extract_keywords(text)
print(keywords)
print(type(keywords))
print(type(keywords[0]))

soo we see that we didnt got words deep learning together, so we can fix that by ngrams

In [ ]:
finalkeyword=kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words=None)
finalkeyword

# NLP-Based Entity Type Classification

This feature uses Natural Language Processing to identify the category of a user-provided entity, such as a programming language, human language, software library, framework, database, or operating system.

In [ ]:
from transformers import pipeline

In [ ]:
entity_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

In [ ]:
entity_categories = [
    "programming language",
    "human language",
    "software library",
    "software framework",
    "database",
    "operating system",
    "development tool"
]

In [ ]:
result = entity_classifier(
    "Python",
    candidate_labels=entity_categories
)

result

In [ ]:
test_entities = [
    "Python",
    "Hindi",
    "NumPy",
    "TensorFlow",
    "MySQL",
    "Linux"
]

for entity in test_entities:
    result = entity_classifier(
        entity,
        candidate_labels=entity_categories
    )

    print(f"{entity} → {result['labels'][0]}")
    print(f"Confidence: {result['scores'][0]:.2%}")
    print()

In [ ]:
test_entities_with_context = [
    "Python is a technology used in computing and software development.",
    "Hindi is a language used by people for communication.",
    "NumPy is a technology used in Python programming and scientific computing.",
    "TensorFlow is a technology used for machine learning development.",
    "MySQL is a technology used for storing and managing data.",
    "Linux is a technology used in computer systems."
]

for text in test_entities_with_context:
    result = entity_classifier(
        text,
        candidate_labels=entity_categories
    )

    print(f"{text}")
    print(f"Prediction: {result['labels'][0]}")
    print(f"Confidence: {result['scores'][0]:.2%}")
    print()

In [10]:
detailed_categories = [
    "a programming language used to write computer programs",
    "a human language used by people to communicate",
    "a software library containing reusable code and functions",
    "a software framework providing a structure for application development",
    "a database system used to store and manage data",
    "an operating system that manages computer hardware and software",
    "a development tool used to build or manage software"
]

In [11]:
result = entity_classifier(
    "NumPy",
    candidate_labels=detailed_categories
)

for label, score in zip(result["labels"], result["scores"]):
    print(f"{label}: {score:.2%}")

a software library containing reusable code and functions: 23.24%
a software framework providing a structure for application development: 21.34%
a programming language used to write computer programs: 19.82%
a development tool used to build or manage software: 18.38%
a database system used to store and manage data: 7.41%
a human language used by people to communicate: 5.35%
an operating system that manages computer hardware and software: 4.46%


In [12]:
test_entities = [
    "Python",
    "Hindi",
    "NumPy",
    "TensorFlow",
    "MySQL",
    "Linux"
]

for entity in test_entities:
    result = entity_classifier(
        entity,
        candidate_labels=detailed_categories
    )

    print(f"{entity}")
    print(f"Prediction: {result['labels'][0]}")
    print(f"Confidence: {result['scores'][0]:.2%}")
    print()

Python
Prediction: a programming language used to write computer programs
Confidence: 64.13%

Hindi
Prediction: a human language used by people to communicate
Confidence: 74.18%

NumPy
Prediction: a software library containing reusable code and functions
Confidence: 23.24%

TensorFlow
Prediction: a software framework providing a structure for application development
Confidence: 37.07%

MySQL
Prediction: a database system used to store and manage data
Confidence: 45.33%

Linux
Prediction: an operating system that manages computer hardware and software
Confidence: 54.02%



In [13]:
category_names = {
    detailed_categories[0]: "Programming Language",
    detailed_categories[1]: "Human Language",
    detailed_categories[2]: "Software Library",
    detailed_categories[3]: "Software Framework",
    detailed_categories[4]: "Database System",
    detailed_categories[5]: "Operating System",
    detailed_categories[6]: "Development Tool"
}

def classify_entity(entity):
    result = entity_classifier(
        entity,
        candidate_labels=detailed_categories
    )

    best_label = result["labels"][0]
    confidence = result["scores"][0]

    return {
        "entity": entity,
        "category": category_names[best_label],
        "confidence": confidence
    }

In [14]:
classification = classify_entity("NumPy")

print("Entity:", classification["entity"])
print("Category:", classification["category"])
print(f"Confidence: {classification['confidence']:.2%}")

Entity: NumPy
Category: Software Library
Confidence: 23.24%


In [15]:
unseen_entities = [
    "Java",
    "Punjabi",
    "Pandas",
    "Django",
    "PostgreSQL",
    "Ubuntu",
    "Git"
]

for entity in unseen_entities:
    classification = classify_entity(entity)

    print(f"{entity} → {classification['category']}")
    print(f"Confidence: {classification['confidence']:.2%}")
    print()

Java → Programming Language
Confidence: 61.74%

Punjabi → Human Language
Confidence: 72.84%

Pandas → Database System
Confidence: 29.01%

Django → Programming Language
Confidence: 23.58%

PostgreSQL → Database System
Confidence: 68.53%

Ubuntu → Operating System
Confidence: 41.21%

Git → Development Tool
Confidence: 35.96%



In [30]:
category_examples = {
    "Programming Language": """
    Python is a programming language.
    Java is a programming language.
    C++ is a programming language.
    JavaScript is a programming language.
    Rust, Go, Swift and Kotlin are programming languages.
    Programming languages are used to write computer programs.
    """,

    "Human Language": """
    Hindi, English, Punjabi, Spanish, French, German and Japanese
    are human languages used by people for communication.
    """,

    "Software Library": """
    NumPy, Pandas, SciPy, Matplotlib, Seaborn and OpenCV
    are software libraries containing reusable code and functions.
    """,

    "Software Framework": """
    TensorFlow, Django, Flask, PyTorch, React, Angular and Spring
    are software frameworks used to build applications and systems.
    """,

    "Database System": """
    MySQL, PostgreSQL, MongoDB, SQLite, Oracle and Redis
    are database systems used to store and manage data.
    """,

    "Operating System": """
    Linux, Ubuntu, Windows, macOS, Android and Fedora
    are operating systems that manage computer hardware and software.
    """,

    "Development Tool": """
    Git, Docker, Jenkins, Maven, Gradle and Kubernetes
    are development tools used to build, manage or deploy software.
    """
}

In [31]:
from sentence_transformers import SentenceTransformer

semantic_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

category_names_list = list(category_examples.keys())
category_texts = list(category_examples.values())

category_embeddings = semantic_model.encode(
    category_texts,
    normalize_embeddings=True
)


print("Categories:", category_names_list)
print("Embedding shape:", category_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Categories: ['Programming Language', 'Human Language', 'Software Library', 'Software Framework', 'Database System', 'Operating System', 'Development Tool']
Embedding shape: (7, 384)


In [22]:
import numpy as np

def classify_entity_semantic(entity):
    entity_embedding = semantic_model.encode(
        [entity],
        normalize_embeddings=True
    )

    similarity_scores = np.dot(
        category_embeddings,
        entity_embedding[0]
    )

    best_index = np.argmax(similarity_scores)

    return {
        "entity": entity,
        "category": category_names_list[best_index],
        "similarity": similarity_scores[best_index]
    }

In [23]:
test_entity = classify_entity_semantic("Pandas")

print("Entity:", test_entity["entity"])
print("Category:", test_entity["category"])
print(f"Similarity: {test_entity['similarity']:.2%}")

Entity: Pandas
Category: Software Library
Similarity: 38.19%


In [24]:
truly_unseen_entities = [
    "Ruby",
    "Bengali",
    "Scikit-learn",
    "Laravel",
    "Cassandra",
    "Debian",
    "Ansible"
]

for entity in truly_unseen_entities:
    result = classify_entity_semantic(entity)

    print(f"{entity} → {result['category']}")
    print(f"Similarity: {result['similarity']:.2%}")
    print()

Ruby → Programming Language
Similarity: 27.18%

Bengali → Human Language
Similarity: 38.10%

Scikit-learn → Software Library
Similarity: 43.12%

Laravel → Database System
Similarity: 20.74%

Cassandra → Database System
Similarity: 24.23%

Debian → Operating System
Similarity: 28.37%

Ansible → Development Tool
Similarity: 31.04%



In [25]:
def predict_entity_type(entity):
    result = classify_entity_semantic(entity)

    return {
        "entity": result["entity"],
        "type": result["category"],
        "score": round(float(result["similarity"]), 4)
    }

In [26]:
predict_entity_type("NumPy")

{'entity': 'NumPy', 'type': 'Software Library', 'score': 0.4178}

In [27]:
def entity_type_detector():
    entity = input("Enter a language, library, framework, database, OS, or tool: ")

    result = predict_entity_type(entity)

    print("\n--- NLP Entity Classification Result ---")
    print("Entity:", result["entity"])
    print("Detected Type:", result["type"])
    print("Similarity Score:", result["score"])

In [28]:
entity_type_detector()

Enter a language, library, framework, database, OS, or tool: Django

--- NLP Entity Classification Result ---
Entity: Django
Detected Type: Software Framework
Similarity Score: 0.4494


In [33]:
final_test_entities = [
    "Python",
    "Hindi",
    "NumPy",
    "Django",
    "PostgreSQL",
    "Ubuntu",
    "Git"
]

print("=" * 50)
print("NLP-BASED SEMANTIC ENTITY TYPE CLASSIFIER")
print("=" * 50)

for entity in final_test_entities:
    result = predict_entity_type(entity)

    print(f"\nEntity: {result['entity']}")
    print(f"Detected Type: {result['type']}")
    print(f"Similarity Score: {result['score']}")

NLP-BASED SEMANTIC ENTITY TYPE CLASSIFIER

Entity: Python
Detected Type: Programming Language
Similarity Score: 0.5014

Entity: Hindi
Detected Type: Human Language
Similarity Score: 0.5363

Entity: NumPy
Detected Type: Software Library
Similarity Score: 0.4178

Entity: Django
Detected Type: Software Framework
Similarity Score: 0.4494

Entity: PostgreSQL
Detected Type: Database System
Similarity Score: 0.5026

Entity: Ubuntu
Detected Type: Operating System
Similarity Score: 0.4749

Entity: Git
Detected Type: Development Tool
Similarity Score: 0.5097
